# Deep Learning Tutorial: MLPs, CNNs and Data Augmentation

================================================================================

This notebook covers Multi-Layer Perceptrons (MLPs), Convolutional Neural Networks (CNNs), and data augmentation techniques for deep learning with PyTorch.

## Topics Covered:
1. **Multi-Layer Perceptrons (MLPs)** - Building fully-connected neural networks
2. **Convolutions for Images** - Understanding how convolutions work
3. **Convolutional Neural Networks (CNNs)** - Building and training CNNs
4. **Data Augmentation Techniques** - Improving model generalization

**Author:** Compiled from CPS-470 course materials

================================================================================

## Import Required Libraries

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

---

# SECTION 5: MULTI-LAYER PERCEPTRONS (MLPs)

Multi-layer perceptrons are fully-connected neural networks with multiple hidden layers. They can approximate any continuous function (universal approximation theorem) given enough hidden units.

PyTorch's nn.Module provides convenient building blocks for creating MLPs.

**Key components of an MLP:**
- `nn.Linear`: fully-connected layer (matrix multiplication + bias)
- `nn.ReLU`: activation function (adds nonlinearity)
- `nn.Sequential`: chains layers together

**Design choices:**
- Number of layers (depth): deeper networks can learn more complex patterns
- Number of neurons per layer (width): wider layers have more capacity
- Activation functions: ReLU is most common for hidden layers

**Trade-offs:**
- More parameters = more capacity but also more risk of overfitting
- Deeper networks can be harder to train (vanishing gradients)

---

### Helper Functions

In [ ]:
def load_california_housing_data():
    """Load California Housing dataset as float32 tensors."""
    data = fetch_california_housing()
    X = torch.tensor(data.data, dtype=torch.float32)
    y = torch.tensor(data.target, dtype=torch.float32).unsqueeze(1)
    return X, y


def train_and_evaluate_mlp(model, X, y, lr=0.01, epochs=100):
    """Helper function to train and evaluate an MLP."""
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    
    for epoch in range(100):
        predictions = model(X)
        loss = loss_fn(predictions, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    with torch.no_grad():
        final_loss = loss_fn(model(X), y).item()
    
    print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Final MSE: {final_loss:.4f}")


print("Helper functions defined successfully!")

### 5.1: Load and Normalize Data

In [ ]:
print("\n" + "=" * 80)
print("SECTION 5: MULTI-LAYER PERCEPTRONS")
print("=" * 80)

print("\n5.1: California Housing Dataset")
print("-" * 40)

X, y = load_california_housing_data()

print(f"Dataset: {X.shape[0]} houses")
print(f"Features: {X.shape[1]} (median income, house age, rooms, etc.)")

# Normalize features to mean 0, std 1
# This helps gradient descent converge faster
mean = X.mean(dim=0)
std = X.std(dim=0)
X = (X - mean) / std

print(f"After normalization:")
print(f"  Feature means: {X.mean(dim=0)[:3].tolist()} ...")
print(f"  Feature stds: {X.std(dim=0)[:3].tolist()} ...")

### 5.2: Build MLP with nn.Sequential

In [ ]:
print("\n5.2: Building MLPs with Different Architectures")
print("-" * 40)

# Architecture 1: 8 -> 128 -> 64 -> 1
model1 = nn.Sequential(
    nn.Linear(8, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

print("Model 1 architecture: 8 -> 128 -> 64 -> 1")
print(f"  Total parameters: {sum(p.numel() for p in model1.parameters()):,}")

# Train model 1
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model1.parameters(), lr=0.01)

print("  Training for 100 epochs...")
for epoch in range(100):
    predictions = model1(X)
    loss = loss_fn(predictions, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    final_loss_1 = loss_fn(model1(X), y).item()
print(f"  Final MSE: {final_loss_1:.4f}")

### 5.3: Experiment with Depth and Width

In [ ]:
print("\n5.3: Comparing Different Architectures")
print("-" * 40)

# Architecture 2: Increasing width (8 -> 32 -> 64 -> 128 -> 1)
model2 = nn.Sequential(
    nn.Linear(8, 32),
    nn.ReLU(),
    nn.Linear(32, 64),
    nn.ReLU(),
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, 1)
)

print("Model 2 (increasing width): 8 -> 32 -> 64 -> 128 -> 1")
train_and_evaluate_mlp(model2, X, y, lr=0.01, epochs=100)

# Architecture 3: Large pyramid (8 -> 256 -> 128 -> 64 -> 32 -> 1)
model3 = nn.Sequential(
    nn.Linear(8, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

print("\nModel 3 (large pyramid): 8 -> 256 -> 128 -> 64 -> 32 -> 1")
train_and_evaluate_mlp(model3, X, y, lr=0.01, epochs=100)

---

# SECTION 7: CONVOLUTIONS FOR IMAGES

Convolutional layers are fundamental to processing images. Unlike fully-connected layers, convolutions:
- Preserve spatial structure
- Share weights across locations (translation invariance)
- Have far fewer parameters

A convolution slides a small filter (kernel) across the image, computing dot products at each position to detect local patterns.

---

### 7.1: Load MNIST Dataset

In [ ]:
"""
SECTION 7: Convolutions for Images

Convolution operation:
- Input: image (H x W)
- Kernel: small filter (K x K), typically 3x3 or 5x5
- Output: feature map (H-K+1 x W-K+1)

How it works:
1. Place kernel at top-left of image
2. Compute element-wise product and sum (dot product)
3. This gives one output value
4. Slide kernel right by 1 pixel and repeat
5. When row is done, move to next row

Different kernels detect different features:
- Edge detectors: highlight boundaries
- Blur filters: smooth the image
- Sharpening filters: enhance details
"""

print("\n" + "=" * 80)
print("SECTION 7: CONVOLUTIONS FOR IMAGES")
print("=" * 80)

# Load MNIST dataset
print("\n7.1: Loading MNIST Handwritten Digits")
print("-" * 40)

transform = transforms.ToTensor()
mnist = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

print(f"MNIST dataset: {len(mnist)} training images")
print("Image size: 28x28 pixels, grayscale")

# Get first image
image, label = mnist[0]
image = image.squeeze(0)  # Remove channel dimension

print(f"\nSample image:")
print(f"  Label: {label}")
print(f"  Shape: {tuple(image.shape)}")
print(f"  Value range: [{image.min():.2f}, {image.max():.2f}]")

### 7.2: Implement Convolution from Scratch

In [ ]:
print("\n7.2: Implementing 2D Convolution")
print("-" * 40)

def my_conv2d(image, kernel):
    """
    Perform 2D convolution using nested loops.
    
    Args:
        image: (H, W) tensor
        kernel: (K, K) tensor
    
    Returns:
        output: (H-K+1, W-K+1) tensor
    """
    H, W = image.shape
    K = kernel.shape[0]
    out_h = H - K + 1
    out_w = W - K + 1
    out = torch.zeros((out_h, out_w), dtype=image.dtype)
    
    # Slide kernel across image
    for i in range(out_h):
        for j in range(out_w):
            # Extract image patch
            patch = image[i:i+K, j:j+K]
            # Compute dot product with kernel
            out[i, j] = (patch * kernel).sum()
    
    return out

# Test with a simple kernel
kernel = torch.tensor([[0.2341, 0.5123, 0.9812],
                      [0.1011, 0.7765, 0.3341],
                      [0.6543, 0.0987, 0.4432]])

print("Testing convolution implementation...")
my_out = my_conv2d(image, kernel)

# Compare with PyTorch's conv2d
torch_out = F.conv2d(
    image.unsqueeze(0).unsqueeze(0),  # Add batch and channel dims
    kernel.unsqueeze(0).unsqueeze(0),
).squeeze()

print(f"  Input shape: {tuple(image.shape)}")
print(f"  Kernel shape: {tuple(kernel.shape)}")
print(f"  Output shape: {tuple(my_out.shape)}")
print(f"  Max difference from PyTorch: {(my_out - torch_out).abs().max().item():.6f}")

### 7.3: Exploring Different Kernels

In [ ]:
print("\n7.3: Different Kernels Detect Different Features")
print("-" * 40)

# Define various kernels
kernels = {
    "Vertical edges": torch.tensor([[-1., 0., 1.],
                                   [-1., 0., 1.],
                                   [-1., 0., 1.]]),
    
    "Horizontal edges": torch.tensor([[-1., -1., -1.],
                                      [0., 0., 0.],
                                      [1., 1., 1.]]),
    
    "Box blur": torch.ones(3, 3) / 9,
    
    "Gaussian blur": torch.tensor([[1., 2., 1.],
                                   [2., 4., 2.],
                                   [1., 2., 1.]]) / 16,
    
    "Sharpen": torch.tensor([[0., -1., 0.],
                            [-1., 5., -1.],
                            [0., -1., 0.]]),
    
    "45° edges": torch.tensor([[0., 1., 2.],
                               [-1., 0., 1.],
                               [-2., -1., 0.]]),
}

print("Applying different kernels to detect various features:")
for name, kernel in kernels.items():
    output = my_conv2d(image, kernel)
    print(f"\n  {name}:")
    print(f"    Output range: [{output.min():.2f}, {output.max():.2f}]")
    print(f"    Output mean: {output.mean():.2f}")

### 7.4: Visualize Kernel Effects

In [ ]:
print("\n7.4: Visualizing Kernel Effects on MNIST Image")
print("-" * 40)

# Create a grid to show original image and all kernel effects
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
fig.suptitle('Effects of Different Kernels on MNIST Digit', fontsize=16, fontweight='bold')

# Show original image
axes[0, 0].imshow(image, cmap='gray')
axes[0, 0].set_title('Original Image', fontweight='bold')
axes[0, 0].axis('off')

# Apply and show each kernel effect
kernel_list = list(kernels.items())
for idx, (name, kernel) in enumerate(kernel_list):
    # Calculate position in grid (skip first position which is original)
    row = (idx + 1) // 4
    col = (idx + 1) % 4
    
    # Apply convolution
    output = my_conv2d(image, kernel)
    
    # Display the result
    axes[row, col].imshow(output, cmap='gray')
    axes[row, col].set_title(name, fontweight='bold')
    axes[row, col].axis('off')

# Hide the last empty subplot
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

---

# SECTION 8: CONVOLUTIONAL NEURAL NETWORKS (CNNs)

CNNs stack convolutional layers with pooling to build hierarchical representations:
- Early layers: detect simple features (edges, corners)
- Middle layers: combine into parts (eyes, wheels)
- Late layers: recognize whole objects (faces, cars)

Pooling (max or average) reduces spatial dimensions while keeping important features, providing translation invariance.

---

### 8.1: Setting Up MNIST Data Loaders

In [ ]:
"""
SECTION 8: Convolutional Neural Networks

CNN architecture pattern:
1. Convolution: detect local features
2. Activation (ReLU): add nonlinearity
3. Pooling: downsample, reduce parameters
4. Repeat: build deeper representations
5. Flatten: convert to vector
6. Fully-connected: final classification

LeNet-5 architecture (classic CNN for digits):
- Input: 28x28 grayscale image
- Conv1: 1 -> 6 channels, 5x5 kernel -> 28x28x6
- Pool1: 2x2 max pooling -> 14x14x6
- Conv2: 6 -> 16 channels, 5x5 kernel -> 14x14x16
- Pool2: 2x2 max pooling -> 7x7x16
- Flatten: 7x7x16 = 784 features
- FC1: 784 -> 120
- FC2: 120 -> 84
- FC3: 84 -> 10 (output classes)
"""

print("\n" + "=" * 80)
print("SECTION 8: CONVOLUTIONAL NEURAL NETWORKS")
print("=" * 80)

print("\n8.1: Preparing MNIST Dataset")
print("-" * 40)

transform = transforms.ToTensor()
full_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Split into train and validation
train_dataset, val_dataset = random_split(full_dataset, [50000, 10000])

# Create data loaders for batching
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: 64")

### 8.2: Building LeNet-5 CNN

In [ ]:
print("\n8.2: LeNet-5 Architecture")
print("-" * 40)

model = nn.Sequential(
    # First convolutional block
    nn.Conv2d(1, 6, kernel_size=5, padding=2),  # 28x28x1 -> 28x28x6
    nn.ReLU(),
    nn.MaxPool2d(2),  # 28x28x6 -> 14x14x6
    
    # Second convolutional block
    nn.Conv2d(6, 16, kernel_size=5, padding=2),  # 14x14x6 -> 14x14x16
    nn.ReLU(),
    nn.MaxPool2d(2),  # 14x14x16 -> 7x7x16
    
    # Flatten and fully-connected layers
    nn.Flatten(),  # 7x7x16 = 784
    nn.Linear(784, 120),
    nn.ReLU(),
    nn.Linear(120, 84),
    nn.ReLU(),
    nn.Linear(84, 10)  # 10 digit classes
)

print("LeNet-5 architecture:")
print("  Conv1: 1->6 channels, 5x5 kernel, padding=2")
print("  ReLU + MaxPool(2x2)")
print("  Conv2: 6->16 channels, 5x5 kernel, padding=2")
print("  ReLU + MaxPool(2x2)")
print("  Flatten -> FC(784->120) -> FC(120->84) -> FC(84->10)")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

### 8.3: Training the CNN

In [ ]:
print("\n8.3: Training LeNet-5 on MNIST")
print("-" * 40)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

def train_epoch(model, loader, criterion, optimizer):
    """Train for one epoch."""
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for images, labels in loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    
    return total_loss / total, correct / total

def evaluate(model, loader, criterion):
    """Evaluate on validation/test set."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
    
    return total_loss / total, correct / total

print("Training for 10 epochs...")
start_time = time.time()

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)
    
    print(f"Epoch {epoch+1}/10:")
    print(f"  Train: loss = {train_loss:.4f}, acc = {train_acc:.2%}")
    print(f"  Val:   loss = {val_loss:.4f}, acc = {val_acc:.2%}")

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed:.1f} seconds")

# Final test evaluation
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f"\nFinal Test Accuracy: {test_acc:.2%}")

---

# SECTION 9: DATA AUGMENTATION

Data augmentation artificially increases dataset size by applying random transformations to training images. This:
- Reduces overfitting by exposing model to more variations
- Improves generalization to new data
- Makes model more robust to transformations

Common augmentations:
- Geometric: flips, rotations, scaling, cropping
- Color: brightness, contrast, saturation, hue
- Noise: blur, gaussian noise, cutout

---

### 9.1: Loading CIFAR-10 Dataset

In [ ]:
"""
SECTION 9: Data Augmentation Techniques

Data augmentation applies random transformations during training:
- Each epoch, images look slightly different
- Model must learn features robust to these variations
- Effectively multiplies dataset size

Common transformations:
- RandomHorizontalFlip: mirrors image left-right
- RandomRotation: rotates by random angle
- ColorJitter: changes brightness, contrast, saturation, hue
- RandomAffine: combines translation, rotation, scale, shear
- RandomCrop: extracts random sub-region

Important: Only augment training data, not validation/test!
"""

print("\n" + "=" * 80)
print("SECTION 9: DATA AUGMENTATION")
print("=" * 80)

print("\n9.1: CIFAR-10 Natural Images")
print("-" * 40)

# CIFAR-10: 60k 32x32 color images in 10 classes
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
          'dog', 'frog', 'horse', 'ship', 'truck']

# Load without augmentation first
basic_transform = transforms.ToTensor()
dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=basic_transform)

print(f"CIFAR-10 dataset: {len(dataset)} training images")
print(f"Image size: 32x32 pixels, RGB (3 channels)")
print(f"Classes: {', '.join(CLASSES)}")

# Get a batch of images
loader = DataLoader(dataset, batch_size=16, shuffle=True)
images, labels = next(iter(loader))

print(f"\nSample batch:")
print(f"  Images shape: {tuple(images.shape)}")  # (16, 3, 32, 32)
print(f"  Labels: {[CLASSES[l] for l in labels[:8]]}")

### 9.2: Manual Augmentation Functions

In [ ]:
print("\n9.2: Implementing Simple Augmentations")
print("-" * 40)

def hflip(img):
    """Horizontal flip: mirror image left-right."""
    return torch.flip(img, dims=[2])  # Flip width dimension

def pixel_jitter(img, sigma=0.1):
    """Add random Gaussian noise to each pixel."""
    noise = torch.randn_like(img) * sigma
    return (img + noise).clamp(0, 1)

print("Manual augmentation functions:")
print("  - hflip: mirrors image horizontally")
print("  - pixel_jitter: adds Gaussian noise")

# Apply augmentations
sample_img = images[0]
flipped = hflip(sample_img)
jittered = pixel_jitter(sample_img, sigma=0.1)

print(f"\nOriginal image shape: {tuple(sample_img.shape)}")
print(f"Flipped image shape: {tuple(flipped.shape)}")
print(f"Jittered image range: [{jittered.min():.2f}, {jittered.max():.2f}]")

### 9.2.1: Visualize Manual Augmentations

In [ ]:
print("\n9.2.1: Visualizing Manual Augmentation Effects on CIFAR-10")
print("-" * 40)

# Pick a sample image and apply different augmentations
sample_img = images[0]

# Create different augmented versions
flipped = hflip(sample_img)
jittered_light = pixel_jitter(sample_img, sigma=0.05)
jittered_medium = pixel_jitter(sample_img, sigma=0.1)
jittered_heavy = pixel_jitter(sample_img, sigma=0.2)

# Create visualization
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Manual Augmentation Effects on CIFAR-10 Image', fontsize=14, fontweight='bold')

# Original - use bicubic interpolation for smoother display
axes[0].imshow(sample_img.permute(1, 2, 0), interpolation='bicubic')
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

# Horizontal flip
axes[1].imshow(flipped.permute(1, 2, 0), interpolation='bicubic')
axes[1].set_title('Horizontal Flip', fontweight='bold')
axes[1].axis('off')

# Different jitter levels
axes[2].imshow(jittered_light.permute(1, 2, 0), interpolation='bicubic')
axes[2].set_title('Light Jitter (σ=0.05)', fontweight='bold')
axes[2].axis('off')

axes[3].imshow(jittered_medium.permute(1, 2, 0), interpolation='bicubic')
axes[3].set_title('Medium Jitter (σ=0.1)', fontweight='bold')
axes[3].axis('off')

axes[4].imshow(jittered_heavy.permute(1, 2, 0), interpolation='bicubic')
axes[4].set_title('Heavy Jitter (σ=0.2)', fontweight='bold')
axes[4].axis('off')

plt.tight_layout()
plt.show()

print(f"\nOriginal image class: {CLASSES[labels[0]]}")

### 9.3: Using torchvision Transforms

In [ ]:
print("\n9.3: Comprehensive Augmentation Pipeline")
print("-" * 40)

# Define augmentation pipeline
train_transform = transforms.Compose([
    # Color augmentation
    transforms.ColorJitter(
        brightness=0.5,  # Vary brightness by +/- 50%
        contrast=0.5,    # Vary contrast by +/- 50%
        saturation=0.5,  # Vary saturation by +/- 50%
        hue=0.5          # Vary hue by +/- 50%
    ),
    
    # Geometric augmentation
    transforms.RandomAffine(
        degrees=10,           # Rotate by +/- 10 degrees
        translate=(0.1, 0.1), # Translate by up to 10% of image size
        scale=(0.9, 1.1),     # Scale by 90% to 110%
        shear=10              # Shear by +/- 10 degrees
    ),
    
    # Random horizontal flip
    transforms.RandomHorizontalFlip(p=0.5),
    
    # Convert to tensor
    transforms.ToTensor()
])

# Load augmented dataset
aug_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
aug_loader = DataLoader(aug_dataset, batch_size=16, shuffle=True)
aug_images, aug_labels = next(iter(aug_loader))

print(f"\nAugmented batch shape: {tuple(aug_images.shape)}")
print("Each training epoch will see different augmented versions!")

### 9.3.1: Visualize Comprehensive Augmentation Pipeline

In [ ]:
print("\n9.3.1: Visualizing Comprehensive Augmentation Effects")
print("-" * 40)

# Get an original image (without augmentation) using PIL
from PIL import Image
import numpy as np

# Get original image from the basic dataset
original_dataset = datasets.CIFAR10(root="./data", train=True, download=False, transform=None)
original_img, original_label = original_dataset[0]

print(f"Original image class: {CLASSES[original_label]}")

# Apply the augmentation pipeline multiple times to see variation
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f'Multiple Augmented Versions of the Same {CLASSES[original_label].upper()} Image', 
             fontsize=16, fontweight='bold')

# Show original image with bicubic interpolation for smoother display
axes[0, 0].imshow(original_img, interpolation='bicubic')
axes[0, 0].set_title('Original Image', fontweight='bold', fontsize=12)
axes[0, 0].axis('off')

# Generate 7 different augmented versions
for idx in range(7):
    row = (idx + 1) // 4
    col = (idx + 1) % 4
    
    # Apply augmentation
    augmented = train_transform(original_img)
    
    # Convert tensor to displayable format
    augmented_display = augmented.permute(1, 2, 0).clamp(0, 1)
    
    # Use bicubic interpolation for smoother display
    axes[row, col].imshow(augmented_display, interpolation='bicubic')
    axes[row, col].set_title(f'Augmented #{idx+1}', fontweight='bold', fontsize=12)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()